In [1]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from QuantNado.combine_metadata import combine_metadata_files
from QuantNado.make_zarr_store import bams_to_zarr

# Make Zarr Dataset
This notebook makes, loads, and explores the Zarr xarray dataset created from BAM files.

In [2]:
bam_files = sorted(
    glob.glob("data/2025-12-17_menin_inh_24hr/seqnado_output/**/aligned/SEM*.bam")
)
chrom_sizes_path = "data/hg38/hg38.chrom.sizes"

print(f"Found {len(bam_files)} BAM files")
print(f"Chromosome sizes: {chrom_sizes_path}")

Found 20 BAM files
Chromosome sizes: data/hg38/hg38.chrom.sizes


## Make combined metadata

In [3]:
metadata_files = sorted(glob.glob("data/2025-12-17_menin_inh_24hr/metadata_*.csv"))
metadata_path = Path("example/combined_metadata.csv")

combine_metadata = combine_metadata_files(
    metadata_files=metadata_files, 
    output_path=metadata_path
)

2025-12-19 17:28:33.657 | INFO     | QuantNado.combine_metadata:combine_metadata_files:36 - Reading metadata file: data/2025-12-17_menin_inh_24hr/metadata_atac.csv
2025-12-19 17:28:33.659 | INFO     | QuantNado.combine_metadata:combine_metadata_files:36 - Reading metadata file: data/2025-12-17_menin_inh_24hr/metadata_chip.csv
2025-12-19 17:28:33.660 | INFO     | QuantNado.combine_metadata:combine_metadata_files:36 - Reading metadata file: data/2025-12-17_menin_inh_24hr/metadata_rna.csv
2025-12-19 17:28:33.664 | INFO     | QuantNado.combine_metadata:combine_metadata_files:71 - Combined metadata saved to example/combined_metadata.csv
2025-12-19 17:28:33.664 | INFO     | QuantNado.combine_metadata:combine_metadata_files:72 - Total samples: 14
2025-12-19 17:28:33.665 | INFO     | QuantNado.combine_metadata:combine_metadata_files:74 - Samples by assay:
assay
ChIP    6
RNA     6
ATAC    2
Name: count, dtype: int64


## Process BAM files to Zarr

Process each BAM file using the parallel chromosome processing:

In [ ]:
output_path = Path("example/dataset_menin_inh_24hr.zarr")

bams_to_zarr(
    bam_files = bam_files,
    chromsizes = chrom_sizes_path,
    store_path = output_path,
    filter_chromosomes = True,
    max_workers = 10,
    overwrite = True,
    metadata = metadata_path,
    use_zip = True,
    group_by_assay = True
)

2025-12-19 17:28:33.669 | WARNING  | QuantNado.make_zarr_store:bams_to_zarr:558 - Deleting existing Zarr store at: example/dataset_menin_inh_24hr.zarr.tmp
2025-12-19 17:28:33.865 | INFO     | QuantNado.make_zarr_store:_parse_chromsizes:143 - Loaded 25 chromosomes from data/hg38/hg38.chrom.sizes
2025-12-19 17:28:33.866 | INFO     | QuantNado.make_zarr_store:bams_to_zarr:569 - Loading metadata from example/combined_metadata.csv
2025-12-19 17:28:33.868 | INFO     | QuantNado.make_zarr_store:bams_to_zarr:593 - Processing 20 BAM files into unified dataset: 'example/dataset_menin_inh_24hr.zarr'
2025-12-19 17:28:33.872 | INFO     | QuantNado.make_zarr_store:_process_assay:377 - Found 3 assays: ['ATAC', 'ChIP', 'RNA']
2025-12-19 17:28:33.873 | INFO     | QuantNado.make_zarr_store:_process_assay:381 - Total samples across all assays: 20
2025-12-19 17:28:33.873 | INFO     | QuantNado.make_zarr_store:_process_assay:383 -   ATAC: 2 samples
2025-12-19 17:28:33.873 | INFO     | QuantNado.make_zarr_s

# Load Dataset

In [ ]:
output_path = Path("example/dataset_menin_inh_24hr.zarr")
zarr.consolidate_metadata("example/dataset_menin_inh_24hr.zarr")
ds = xr.open_zarr(output_path)

# Explore Zarr Dataset



In [ ]:
ds

In [ ]:
# Check dimensions and coordinates
print("Dimensions:", ds.dims)
print("\nCoordinates:", list(ds.coords))
print("\nData variables:", list(ds.data_vars))
print("\nAttributes:", ds.attrs)

## PCA analysis

Perform Principal Component Analysis on the entire dataset to visualize sample relationships:

In [ ]:
def prepare_pca_data(ds, subsample_positions=10000):
    """
    Prepare dataset for PCA by flattening and subsampling.

    Parameters:
    - ds: xarray Dataset with signal
    - subsample_positions: Number of random positions to sample (reduces memory)

    Returns:
    - signal_flat: Array for PCA (samples x subsampled features)
    """
    signal_data = ds["signal"]
    n_samples = signal_data.sizes["sample"]
    n_chroms = signal_data.sizes["chromosome"]
    n_positions = signal_data.sizes["position"]

    print(f"Original shape: {n_samples} samples x {n_chroms} chromosomes x {n_positions} positions")

    # Stack chromosome and position into a single feature dimension
    stacked = signal_data.stack(feature=("chromosome", "position"))

    # Subsample features (positions) efficiently
    n_features = stacked.sizes["feature"]
    if n_features > subsample_positions:
        print(f"Subsampling to {subsample_positions} random positions...")
        np.random.seed(42)
        subsample_idx = np.random.choice(n_features, subsample_positions, replace=False)
        stacked = stacked.isel(feature=subsample_idx)

    print(f"Flattened shape: {stacked.shape}")

    # Convert to numpy array for PCA
    signal_flat = stacked.values

    return signal_flat


def compute_pca(signal_flat, n_components=5):
    """
    Compute PCA on flattened signal data.

    Parameters:
    - signal_flat: Array of shape (n_samples, n_features)
    - n_components: Number of principal components to compute

    Returns:
    - pca_result: PCA-transformed data
    - pca: PCA object with explained variance
    """
    print("\nStandardizing data...")
    scaler = StandardScaler()
    signal_scaled = scaler.fit_transform(signal_flat)

    print(f"Computing PCA with {n_components} components...")
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(signal_scaled)

    print("\nExplained variance ratio:")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"  PC{i + 1}: {var * 100:.2f}%")
    print(f"  Total: {pca.explained_variance_ratio_.sum() * 100:.2f}%")

    return pca_result, pca


In [ ]:
# Prepare and compute PCA
print("Preparing data for PCA...")
signal_flat = prepare_pca_data(ds, subsample_positions=50000)
pca_result, pca = compute_pca(signal_flat, n_components=5)
print(f"\nPCA result shape: {pca_result.shape}")

### Plot PCA results

Visualize samples in PC space:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: PC1 vs PC2
ax = axes[0]
scatter = ax.scatter(
    pca_result[:, 0],
    pca_result[:, 1],
    s=100,
    alpha=0.7,
    edgecolors="black",
    linewidth=1,
)

# Add sample labels
for i, sample in enumerate(ds.sample.values):
    ax.annotate(
        sample,
        (pca_result[i, 0], pca_result[i, 1]),
        fontsize=8,
        alpha=0.8,
        xytext=(5, 5),
        textcoords="offset points",
    )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
ax.set_title("PCA: PC1 vs PC2")
ax.grid(True, alpha=0.3)

# Plot 2: Scree plot (variance explained)
ax = axes[1]
pc_labels = [f"PC{i + 1}" for i in range(len(pca.explained_variance_ratio_))]
ax.bar(pc_labels, pca.explained_variance_ratio_ * 100, alpha=0.7, edgecolor="black")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Variance Explained (%)")
ax.set_title("Scree Plot")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# Create PCA DataFrame for easier analysis
pca_df = pd.DataFrame(
    pca_result,
    columns=[f"PC{i + 1}" for i in range(pca_result.shape[1])],
    index=ds.sample.values,
)

print("\nPCA coordinates:")
print(pca_df)

### PCA with metadata coloring

If your dataset has metadata coordinates, you can color points by groups:

In [ ]:
# Check if we have metadata to use for coloring
metadata_coords = [
    c for c in ds.coords if c not in ["sample", "chromosome", "position"]
]

if metadata_coords:
    print(f"Available metadata for coloring: {metadata_coords}")

    # Example: Color by first metadata coordinate
    color_by = metadata_coords[0]
    colors = ds.coords[color_by].values

    fig, ax = plt.subplots(figsize=(10, 8))

    # Check if colors are categorical or numeric
    if np.issubdtype(colors.dtype, np.number):
        # Numeric coloring
        scatter = ax.scatter(
            pca_result[:, 0],
            pca_result[:, 1],
            c=colors,
            s=100,
            alpha=0.7,
            cmap="viridis",
            edgecolors="black",
            linewidth=1,
        )
        plt.colorbar(scatter, ax=ax, label=color_by)
    else:
        # Categorical coloring
        unique_vals = np.unique(colors)
        cmap = plt.cm.get_cmap("tab10", len(unique_vals))

        for i, val in enumerate(unique_vals):
            mask = colors == val
            ax.scatter(
                pca_result[mask, 0],
                pca_result[mask, 1],
                c=[cmap(i)],
                label=val,
                s=100,
                alpha=0.7,
                edgecolors="black",
                linewidth=1,
            )
        ax.legend(title=color_by, bbox_to_anchor=(1.05, 1), loc="upper left")

    # Add sample labels
    for i, sample in enumerate(ds.sample.values):
        ax.annotate(
            sample,
            (pca_result[i, 0], pca_result[i, 1]),
            fontsize=8,
            alpha=0.8,
            xytext=(5, 5),
            textcoords="offset points",
        )

    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
    ax.set_title(f"PCA colored by {color_by}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No metadata coordinates found for coloring")

# Reduce by ranges

In [ ]:
promoters_bed = "data/hg38/promoters_1024bp.bed"

# Load promoter regions from BED file
promoters_df = pd.read_csv(
    promoters_bed,
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "gene", "score", "strand"],
)

print(f"Loaded {len(promoters_df)} promoter regions")
print(f"Chromosomes: {promoters_df['chrom'].unique()}")
promoters_df.head()

## Extract signal for promoter regions

Now we'll extract the signal values for each promoter region and create a reduced dataset:

In [ ]:
def extract_promoter_signals(ds, promoters_df):
    """
    Extract signal for all promoter regions from the dataset.

    Returns a new dataset with dimensions: (sample, promoter, position)
    where position ranges from 0 to promoter_width for each promoter.
    """
    promoter_signals = []
    valid_promoters = []

    for idx, row in promoters_df.iterrows():
        chrom = row["chrom"]
        start = row["start"]
        end = row["end"]
        gene = row["gene"]

        # Check if chromosome exists in dataset
        if chrom not in ds.chromosome.values:
            continue

        try:
            # Extract signal for this region across all samples
            # Select chromosome and position range
            region_signal = ds.sel(chromosome=chrom, position=slice(start, end))[
                "signal"
            ]

            # Only keep if we got the expected size
            if region_signal.sizes["position"] == (end - start):
                promoter_signals.append(region_signal)
                valid_promoters.append(
                    {
                        "gene": gene,
                        "chrom": chrom,
                        "start": start,
                        "end": end,
                        "strand": row["strand"],
                    }
                )
        except Exception as e:
            # Skip promoters that cause issues
            continue

    if not promoter_signals:
        raise ValueError("No valid promoters found in dataset")

    # Combine all promoter signals
    print(f"Combining {len(promoter_signals)} promoter regions...")
    combined = xr.concat(promoter_signals, dim="promoter")

    # Create a new position coordinate (relative to promoter start)
    promoter_width = promoter_signals[0].sizes["position"]
    combined = combined.assign_coords(position=np.arange(promoter_width))

    # Add promoter metadata as coordinates
    promoter_info_df = pd.DataFrame(valid_promoters)
    combined = combined.assign_coords(
        gene=("promoter", promoter_info_df["gene"].values),
        promoter_chrom=("promoter", promoter_info_df["chrom"].values),
        promoter_start=("promoter", promoter_info_df["start"].values),
        promoter_end=("promoter", promoter_info_df["end"].values),
        promoter_strand=("promoter", promoter_info_df["strand"].values),
    )

    return combined


# Extract promoter signals
print("Extracting promoter signals from dataset...")
promoter_ds = extract_promoter_signals(ds, promoters_df)

print(f"\nPromoter dataset shape: {promoter_ds.shape}")
print(f"Dimensions: {dict(promoter_ds.sizes)}")
print(f"Coordinates: {list(promoter_ds.coords)}")
promoter_ds

## Compute summary statistics

Calculate mean signal across promoters for each sample:

In [ ]:
# Calculate mean signal per promoter (averaged across position)
promoter_means = promoter_ds.mean(dim="position")

print(f"Promoter means shape: {promoter_means.shape}")
print(f"Dimensions: {dict(promoter_means.sizes)}")

# Convert to DataFrame for easier analysis
promoter_summary = promoter_means.to_dataframe().reset_index()
promoter_summary.head(10)

## Visualize promoter signals

Plot average signal across all promoters:

In [ ]:
# Plot average signal profile across all promoters
avg_signal_by_sample = promoter_ds.mean(dim="promoter")

fig, ax = plt.subplots(figsize=(12, 6))

# Plot each sample
for i, sample in enumerate(avg_signal_by_sample.sample.values):
    signal = avg_signal_by_sample.sel(sample=sample)
    ax.plot(signal.position, signal.values, label=sample, alpha=0.7)

ax.set_xlabel("Position relative to promoter start (bp)")
ax.set_ylabel("Average signal")
ax.set_title("Average ChIP-seq signal across all promoters")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Promoter metaplot

Create a metaplot showing average signal across all promoters:

In [ ]:
def create_metaplot(promoter_ds, window_size=1024):
    """
    Create a metaplot showing average signal across all promoters.

    Parameters:
    - promoter_ds: xarray DataArray with promoter signals
    - window_size: Size of the promoter window in bp

    Returns:
    - DataFrame with metaplot data
    """
    # Calculate mean signal across all promoters for each sample
    metaplot_data = promoter_ds.mean(dim="promoter")

    # Convert to DataFrame for plotting
    metaplot_df = metaplot_data.to_dataframe(name="signal").reset_index()

    return metaplot_df


# Create metaplot data
metaplot_df = create_metaplot(promoter_ds)

# Plot metaplot
fig, ax = plt.subplots(figsize=(12, 6))

# Plot each sample
for sample in promoter_ds.sample.values:
    sample_data = metaplot_df[metaplot_df["sample"] == sample]
    ax.plot(
        sample_data["position"],
        sample_data["signal"],
        label=sample,
        alpha=0.7,
        linewidth=2,
    )

# Add TSS marker
tss_position = len(promoter_ds.position) // 2  # Assuming TSS is at center
ax.axvline(
    tss_position, color="black", linestyle="--", linewidth=1.5, alpha=0.5, label="TSS"
)

ax.set_xlabel("Position relative to promoter start (bp)", fontsize=12)
ax.set_ylabel("Average signal", fontsize=12)
ax.set_title(
    "Promoter Metaplot: Average ChIP-seq Signal Across All Promoters",
    fontsize=14,
    fontweight="bold",
)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMetaplot computed across {promoter_ds.sizes['promoter']} promoters")
print(f"Window size: {promoter_ds.sizes['position']} bp")

### Heatmap view of promoter signals

Visualize promoter signals as a heatmap sorted by signal intensity:

In [ ]:
def plot_promoter_heatmap(promoter_ds, sample_idx=0, n_promoters=500, sort_by="mean"):
    """
    Plot heatmap of promoter signals.

    Parameters:
    - promoter_ds: xarray DataArray with promoter signals
    - sample_idx: Index of sample to plot
    - n_promoters: Number of top promoters to show
    - sort_by: How to sort ('mean', 'max', 'sum')
    """
    # Select one sample
    sample_name = promoter_ds.sample.values[sample_idx]
    sample_data = promoter_ds.isel(sample=sample_idx)

    # Calculate sorting metric
    if sort_by == "mean":
        sort_values = sample_data.mean(dim="position")
    elif sort_by == "max":
        sort_values = sample_data.max(dim="position")
    else:  # sum
        sort_values = sample_data.sum(dim="position")

    # Get top N promoters
    top_indices = np.argsort(sort_values.values)[::-1][:n_promoters]
    top_data = sample_data.isel(promoter=top_indices)

    # Get gene names for top promoters
    top_genes = promoter_ds.gene.values[top_indices]

    # Create heatmap
    fig, ax = plt.subplots(figsize=(10, 12))

    im = ax.imshow(
        top_data.values, aspect="auto", cmap="RdBu_r", interpolation="nearest"
    )

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Signal", fontsize=10)

    # Labels
    ax.set_xlabel("Position relative to promoter start (bp)", fontsize=10)
    ax.set_ylabel("Promoters (sorted by signal)", fontsize=10)
    ax.set_title(
        f"Promoter Heatmap: {sample_name}\nTop {n_promoters} promoters by {sort_by} signal",
        fontsize=12,
        fontweight="bold",
    )

    # Add TSS marker
    tss_position = len(promoter_ds.position) // 2
    ax.axvline(tss_position, color="yellow", linestyle="--", linewidth=1.5, alpha=0.8)

    # Set x-axis ticks
    n_ticks = 5
    tick_positions = np.linspace(0, top_data.sizes["position"] - 1, n_ticks).astype(int)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_positions)

    plt.tight_layout()
    plt.show()

    # Show top genes
    print(f"\nTop 20 genes by {sort_by} signal in {sample_name}:")
    for i, gene in enumerate(top_genes[:20], 1):
        print(f"{i:2d}. {gene}")


# Plot heatmap for first sample
plot_promoter_heatmap(promoter_ds, sample_idx=0, n_promoters=500, sort_by="mean")

### Compare metaplots by group

If you have metadata, compare metaplots across experimental groups:

In [ ]:
# Check if we have metadata to group by
metadata_coords = [
    c for c in ds.coords if c not in ["sample", "chromosome", "position"]
]

if metadata_coords:
    print(f"Available metadata for grouping: {metadata_coords}")

    # Example: Group by first metadata coordinate
    group_by = metadata_coords[0]
    groups = ds.coords[group_by].values
    unique_groups = np.unique(groups)

    print(f"\nGrouping samples by: {group_by}")
    print(f"Groups: {unique_groups}")

    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot mean for each group
    for group in unique_groups:
        # Find samples in this group
        group_mask = groups == group
        group_samples = ds.sample.values[group_mask]

        # Select these samples from promoter dataset
        group_indices = [
            np.where(promoter_ds.sample.values == s)[0][0] for s in group_samples
        ]
        group_data = promoter_ds.isel(sample=group_indices)

        # Calculate mean across samples in group, then mean across promoters
        group_metaplot = group_data.mean(dim="sample").mean(dim="promoter")

        # Calculate SEM for error bars
        group_sem = group_data.mean(dim="promoter").std(dim="sample") / np.sqrt(
            len(group_samples)
        )

        # Plot with error bars
        positions = group_metaplot.position.values
        mean_signal = group_metaplot.values

        ax.plot(
            positions,
            mean_signal,
            label=f"{group} (n={len(group_samples)})",
            linewidth=2.5,
            alpha=0.8,
        )
        ax.fill_between(
            positions,
            mean_signal - group_sem.values,
            mean_signal + group_sem.values,
            alpha=0.2,
        )

    # Add TSS marker
    tss_position = len(promoter_ds.position) // 2
    ax.axvline(
        tss_position,
        color="black",
        linestyle="--",
        linewidth=1.5,
        alpha=0.5,
        label="TSS",
    )

    ax.set_xlabel("Position relative to promoter start (bp)", fontsize=12)
    ax.set_ylabel("Average signal", fontsize=12)
    ax.set_title(f"Promoter Metaplot by {group_by}", fontsize=14, fontweight="bold")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No metadata coordinates found for grouping")
    print(
        "\nTo add metadata, include it when combining zarr files using the metadata_df parameter"
    )

## Query specific promoters

Example: Find and plot signal for specific genes:

In [ ]:
# Example: Find promoters for genes of interest
gene_name = "MYC"  # Change this to your gene of interest

# Find matching genes (partial match with startswith)
matching_genes = [g for g in promoter_ds.gene.values if g.startswith(gene_name)]

if matching_genes:
    print(f"Found {len(matching_genes)} genes matching '{gene_name}':")
    print(matching_genes[:10])  # Show first 10

    # Select the first matching gene and plot
    gene_to_plot = matching_genes[0]
    gene_idx = np.where(promoter_ds.gene.values == gene_to_plot)[0][0]
    gene_signal = promoter_ds.isel(promoter=gene_idx)

    fig, ax = plt.subplots(figsize=(12, 6))
    for sample in gene_signal.sample.values:
        signal = gene_signal.sel(sample=sample)
        ax.plot(signal.position, signal.values, label=sample, alpha=0.7)

    ax.set_xlabel("Position relative to promoter start (bp)")
    ax.set_ylabel("Signal")
    ax.set_title(f"ChIP-seq signal at {gene_to_plot} promoter")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f"No genes found matching '{gene_name}'")

## Save promoter dataset

Optionally save the reduced promoter dataset for faster loading:

In [ ]:
# Save promoter dataset
promoter_output = Path("example/promoter_dataset.zarr")

promoter_ds_to_save = promoter_ds.to_dataset(name="signal")

encoding = {
    "signal": {"chunks": (1, 100, 512)}  # (sample, promoter, position)
}

promoter_ds_to_save.to_zarr(
    promoter_output, mode="w", encoding=encoding, consolidated=False
)

print(f"Promoter dataset saved to {promoter_output}")
print(f"Original dataset size: {dict(ds.sizes)}")
print(f"Reduced dataset size: {dict(promoter_ds_to_save.sizes)}")

# Gene-level feature counts from GTF

Extract read counts for gene features from a GTF annotation file:

In [ ]:
def parse_gtf_attributes(attr_string):
    """Parse GTF attribute string into a dictionary."""
    attrs = {}
    for item in attr_string.strip().split(";"):
        item = item.strip()
        if item:
            key_value = item.split(" ", 1)
            if len(key_value) == 2:
                key, value = key_value
                attrs[key] = value.strip('"')
    return attrs


def load_gtf_genes(gtf_file, feature_type="gene"):
    """
    Load gene features from a GTF file.

    Parameters:
    - gtf_file: Path to GTF file
    - feature_type: Feature type to extract ('gene', 'exon', 'transcript', etc.)

    Returns:
    - DataFrame with gene information
    """
    genes = []

    with open(gtf_file, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue

            fields = line.strip().split("\t")
            if len(fields) < 9:
                continue

            chrom, source, ftype, start, end, score, strand, frame, attributes = fields

            if ftype != feature_type:
                continue

            # Parse attributes
            attrs = parse_gtf_attributes(attributes)

            genes.append(
                {
                    "chrom": chrom,
                    "start": int(start) - 1,  # GTF is 1-based, convert to 0-based
                    "end": int(end),
                    "strand": strand,
                    "gene_id": attrs.get("gene_id", ""),
                    "gene_name": attrs.get("gene_name", attrs.get("gene_id", "")),
                    "gene_type": attrs.get("gene_type", attrs.get("gene_biotype", "")),
                }
            )

    return pd.DataFrame(genes)


# GTF file path
gtf_file = "data/hg38/hg38.ncbiRefSeq.gtf"

print(f"Loading genes from GTF: {gtf_file}")
genes_df = load_gtf_genes(gtf_file, feature_type="gene")

print(f"\nLoaded {len(genes_df)} genes")
print(f"Chromosomes: {sorted(genes_df['chrom'].unique())}")
print(f"Gene types: {genes_df['gene_type'].value_counts()[:10]}")
print("\nFirst few genes:")
genes_df.head()

## Compute feature counts

Calculate total read counts for each gene across all samples:

In [ ]:
def compute_feature_counts(ds, genes_df, filter_gene_types=None):
    """
    Compute feature counts (total reads per gene) from the dataset.

    Parameters:
    - ds: xarray Dataset with signal
    - genes_df: DataFrame with gene annotations from GTF
    - filter_gene_types: List of gene types to include (e.g., ['protein_coding'])
                         If None, includes all genes

    Returns:
    - DataFrame with counts (genes x samples)
    """
    # Filter genes by type if requested
    if filter_gene_types:
        genes_df = genes_df[genes_df["gene_type"].isin(filter_gene_types)].copy()
        print(f"Filtering to {len(genes_df)} genes of types: {filter_gene_types}")

    counts_list = []
    valid_genes = []

    print(f"Computing counts for {len(genes_df)} genes...")

    for idx, gene in genes_df.iterrows():
        chrom = gene["chrom"]
        start = gene["start"]
        end = gene["end"]
        gene_id = gene["gene_id"]
        gene_name = gene["gene_name"]

        # Skip if chromosome not in dataset
        if chrom not in ds.chromosome.values:
            continue

        try:
            # Extract signal for this gene region
            gene_signal = ds.sel(chromosome=chrom, position=slice(start, end))["signal"]

            # Sum signal across all positions to get total count per sample
            gene_counts = gene_signal.sum(dim="position")

            counts_list.append(gene_counts.values)
            valid_genes.append(
                {
                    "gene_id": gene_id,
                    "gene_name": gene_name,
                    "gene_type": gene["gene_type"],
                    "chrom": chrom,
                    "start": start,
                    "end": end,
                    "strand": gene["strand"],
                    "length": end - start,
                }
            )

        except Exception as e:
            continue

        # Progress indicator
        if (len(valid_genes) % 1000) == 0:
            print(f"  Processed {len(valid_genes)} genes...")

    print(f"Successfully computed counts for {len(valid_genes)} genes")

    # Create counts DataFrame
    counts_array = np.array(counts_list)  # Shape: (n_genes, n_samples)
    counts_df = pd.DataFrame(counts_array, columns=ds.sample.values)

    # Add gene metadata
    gene_info_df = pd.DataFrame(valid_genes)
    counts_df = pd.concat([gene_info_df, counts_df], axis=1)

    return counts_df


# Compute counts for protein-coding genes
print("Computing feature counts...")
feature_counts = compute_feature_counts(
    ds,
    genes_df,
    filter_gene_types=["protein_coding"],  # Change to None for all gene types
)

print(f"\nFeature counts shape: {feature_counts.shape}")
print(f"Columns: {list(feature_counts.columns)}")
feature_counts.head()

## Normalize counts

Calculate RPKM (Reads Per Kilobase per Million mapped reads):

In [ ]:
def compute_rpkm(counts_df, sample_columns):
    """
    Compute RPKM (Reads Per Kilobase per Million mapped reads).

    RPKM = (counts / gene_length_kb) / (total_counts_millions)

    Parameters:
    - counts_df: DataFrame with counts and 'length' column
    - sample_columns: List of column names containing count data

    Returns:
    - DataFrame with RPKM values
    """
    rpkm_df = counts_df.copy()

    # Gene length in kilobases
    gene_length_kb = counts_df["length"] / 1000

    for sample in sample_columns:
        # Total counts per sample in millions
        total_counts_millions = counts_df[sample].sum() / 1e6

        # Calculate RPKM
        rpkm_df[sample] = (counts_df[sample] / gene_length_kb) / total_counts_millions

    return rpkm_df


# Get sample column names (exclude metadata columns)
metadata_cols = [
    "gene_id",
    "gene_name",
    "gene_type",
    "chrom",
    "start",
    "end",
    "strand",
    "length",
]
sample_cols = [col for col in feature_counts.columns if col not in metadata_cols]

print(f"Computing RPKM for {len(sample_cols)} samples...")
rpkm_df = compute_rpkm(feature_counts, sample_cols)

print(f"\nRPKM shape: {rpkm_df.shape}")
print("\nRPKM values (first few genes):")
rpkm_df.head()

## Save feature counts

Save counts and RPKM to CSV files:

In [ ]:
# Save to CSV
counts_output = Path("example/feature_counts.csv")
rpkm_output = Path("example/feature_counts_rpkm.csv")

feature_counts.to_csv(counts_output, index=False)
rpkm_df.to_csv(rpkm_output, index=False)

print(f"Saved raw counts to: {counts_output}")
print(f"Saved RPKM values to: {rpkm_output}")

# Summary statistics
print(f"\nSummary:")
print(f"  Total genes: {len(feature_counts)}")
print(f"  Total samples: {len(sample_cols)}")
print(f"  Mean counts per gene: {feature_counts[sample_cols].mean(axis=1).mean():.2f}")
print(f"  Mean RPKM per gene: {rpkm_df[sample_cols].mean(axis=1).mean():.2f}")

## Visualize count distribution

Plot distribution of counts across genes and samples:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Total counts per sample
sample_totals = feature_counts[sample_cols].sum(axis=0)
axes[0].bar(range(len(sample_totals)), sample_totals.values)
axes[0].set_xlabel("Sample index")
axes[0].set_ylabel("Total counts")
axes[0].set_title("Total read counts per sample")
axes[0].grid(True, alpha=0.3)

# Plot 2: Distribution of log2(RPKM + 1) for first sample
sample_to_plot = sample_cols[0]
rpkm_values = rpkm_df[sample_to_plot].values
log_rpkm = np.log2(rpkm_values + 1)

axes[1].hist(log_rpkm, bins=50, alpha=0.7, edgecolor="black")
axes[1].set_xlabel("log2(RPKM + 1)")
axes[1].set_ylabel("Number of genes")
axes[1].set_title(f"RPKM distribution: {sample_to_plot}")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show top expressed genes
print("\nTop 10 expressed genes (by mean RPKM):")
mean_rpkm = rpkm_df[sample_cols].mean(axis=1)
top_genes_idx = mean_rpkm.nlargest(10).index
print(rpkm_df.loc[top_genes_idx, ["gene_name", "gene_type"] + sample_cols[:3]])